In [1]:
import pickle

import numpy as np
from sklearn.metrics import matthews_corrcoef, roc_auc_score, accuracy_score, f1_score

In [2]:
def prep_df(df):
    inter_y, sub_y = [], []

    for i, (_, row) in enumerate(df.iterrows()):
        match row["Binding"]:
            case 0:
                inter_y.append(1)
                sub_y.append(0)
            case 1:
                inter_y.append(1)
                sub_y.append(1)
            case 2:
                inter_y.append(0)
            case _:
                raise ValueError("Invalid binding type: " + str(row))
    print("Done loading")
    return np.array(inter_y), np.array(sub_y)

In [3]:
def load_preds(split, seed):
    with open(f'code/rf_{split}_{seed}_preds.pkl', 'rb') as f:
        inter_preds, sub_preds = pickle.load(f)
    return inter_preds, sub_preds

def load_data(split):
    with open(f"data/splits/test_{split}_2S.pkl", "rb") as f:
        test = pickle.load(f)
    test_inter_y, test_sub_y = prep_df(test)
    return test_inter_y, test_sub_y

In [7]:
for split in ["R", "C1e", "C1f", "C1", "C2"]:
    test_inter_y, test_sub_y = load_data(split)
    inter_auroc, inter_mcc, inter_acc, inter_f1 = [], [], [], []
    sub_auroc, sub_mcc, sub_acc, sub_f1 = [], [], [], []
    print(f"Results for split {split}")
    for seed in [42, 123, 456, 789, 999]:
        inter_preds, sub_preds = load_preds(split, seed)

        inter_auroc.append(roc_auc_score(test_inter_y, inter_preds))
        inter_mcc.append(matthews_corrcoef(test_inter_y, inter_preds > 0.5))
        inter_acc.append(accuracy_score(test_inter_y, inter_preds > 0.5))
        inter_f1.append(f1_score(test_inter_y, inter_preds > 0.5))

        sub_auroc.append(roc_auc_score(test_sub_y, sub_preds))
        sub_mcc.append(matthews_corrcoef(test_sub_y, sub_preds > 0.5))
        sub_acc.append(accuracy_score(test_sub_y, sub_preds > 0.5))
        sub_f1.append(f1_score(test_sub_y, sub_preds > 0.5))
    
    print(f"Interactions:")
    print(f"AUROC: {np.mean(inter_auroc):.3f} ± {np.std(inter_auroc):.3f}, ACC: {np.mean(inter_acc):.3f} ± {np.std(inter_acc):.3f}, F1: {np.mean(inter_f1):.3f} ± {np.std(inter_f1):.3f}, MCC: {np.mean(inter_mcc):.3f} ± {np.std(inter_mcc):.3f}")
    print(f"Subclass:")
    print(f"AUROC: {np.mean(sub_auroc):.3f} ± {np.std(sub_auroc):.3f}, ACC: {np.mean(sub_acc):.3f} ± {np.std(sub_acc):.3f}, F1: {np.mean(sub_f1):.3f} ± {np.std(sub_f1):.3f}, MCC: {np.mean(sub_mcc):.3f} ± {np.std(sub_mcc):.3f}")
    print()

Done loading
Results for split R
Interactions:
AUROC: 0.916 ± 0.000, ACC: 0.916 ± 0.000, F1: 0.910 ± 0.000, MCC: 0.831 ± 0.001
Subclass:
AUROC: 0.957 ± 0.001, ACC: 0.957 ± 0.001, F1: 0.958 ± 0.001, MCC: 0.914 ± 0.001

Done loading
Results for split C1e
Interactions:
AUROC: 0.645 ± 0.019, ACC: 0.644 ± 0.019, F1: 0.680 ± 0.015, MCC: 0.299 ± 0.038
Subclass:
AUROC: 0.792 ± 0.010, ACC: 0.790 ± 0.011, F1: 0.805 ± 0.013, MCC: 0.581 ± 0.021

Done loading
Results for split C1f
Interactions:
AUROC: 0.879 ± 0.002, ACC: 0.878 ± 0.002, F1: 0.871 ± 0.002, MCC: 0.756 ± 0.003
Subclass:
AUROC: 0.937 ± 0.001, ACC: 0.937 ± 0.001, F1: 0.940 ± 0.001, MCC: 0.874 ± 0.001

Done loading
Results for split C1
Interactions:
AUROC: 0.740 ± 0.004, ACC: 0.747 ± 0.005, F1: 0.706 ± 0.004, MCC: 0.487 ± 0.010
Subclass:
AUROC: 0.589 ± 0.005, ACC: 0.584 ± 0.006, F1: 0.346 ± 0.014, MCC: 0.264 ± 0.011

Done loading
Results for split C2
Interactions:
AUROC: 0.645 ± 0.016, ACC: 0.653 ± 0.015, F1: 0.698 ± 0.010, MCC: 0.296 ± 0

In [7]:
(inter_preds, inter_label), (sub_preds, sub_label) = load_data('R')
print("Interaction AUC:", roc_auc_score(inter_label, inter_preds))
print("Interaction ACC:", accuracy_score(inter_label, inter_preds > 0.5))
print("Interaction F1:", f1_score(inter_label, inter_preds > 0.5))
print("Interaction MCC:", matthews_corrcoef(inter_label, inter_preds > 0.5))
print()
print("Subclass AUC:", roc_auc_score(sub_label, sub_preds))
print("Subclass ACC:", accuracy_score(sub_label, sub_preds > 0.5))
print("Subclass F1:", f1_score(sub_label, sub_preds > 0.5))
print("Subclass MCC:", matthews_corrcoef(sub_label, sub_preds > 0.5))

Done loading
Interaction AUC: 0.9163210058715203
Interaction ACC: 0.916012905416879
Interaction F1: 0.9095299067130053
Interaction MCC: 0.8313677900489163

Subclass AUC: 0.9569573245475658
Subclass ACC: 0.957146029161424
Subclass F1: 0.9582702702702702
Subclass MCC: 0.9145970889958043


In [8]:
(inter_preds, inter_label), (sub_preds, sub_label) = load_data('C2')
print("Interaction AUC:", roc_auc_score(inter_label, inter_preds))
print("Interaction ACC:", accuracy_score(inter_label, inter_preds > 0.5))
print("Interaction F1:", f1_score(inter_label, inter_preds > 0.5))
print("Interaction MCC:", matthews_corrcoef(inter_label, inter_preds > 0.5))
print()
print("Subclass AUC:", roc_auc_score(sub_label, sub_preds))
print("Subclass ACC:", accuracy_score(sub_label, sub_preds > 0.5))
print("Subclass F1:", f1_score(sub_label, sub_preds > 0.5))
print("Subclass MCC:", matthews_corrcoef(sub_label, sub_preds > 0.5))

Done loading
Interaction AUC: 0.6275199711242851
Interaction ACC: 0.6359510041658895
Interaction F1: 0.6848931704429255
Interaction MCC: 0.26123289757790125

Subclass AUC: 0.7763457619508152
Subclass ACC: 0.7578268876611418
Subclass F1: 0.7494641581328888
Subclass MCC: 0.5591074787761315
